In [ ]:
!pip install groq -q

import json
import time
from groq import Groq
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

{
  "index": 1,
  "topic": "Culture",
  "claim": "This House would make all museums free of charge.",
  "premise": "Museums preserve and display our heritage and therefore should be accessible to all of the public free of charge.",
  "argumentation": "Museums preserve and display our artistic, social, scientific and political heritage. Everyone should have access to such important cultural resources as part of active citizenship and because of the educational opportunities they offer to people of every age. Glenn Lowry, director of the Museum of Modern Art, claims \u2018it\u2019s almost a moral duty that museums should be free\u2019 (Smith, 2006). If museums are not funded sufficiently by the government, they will be forced to charge for entry, and this will inevitably deter many potential visitors, especially the poor and those whose educational and cultural opportunities have already been limited."
}


Claim-Only Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: A just society would ensure that all citizens have equal access to cultural and educational resources, which museums provide. From behind the veil of ignorance, individuals would want to ensure that their opportunities for enrichment and understanding of the world are not limited by economic status. Making museums free of charge promotes fair equality of opportunity and allows the least advantaged to benefit from cultural institutions. This policy aligns with the principle of justice as fairness, which prioritizes the well-being of the least advantaged members of society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against

Argument: Making museums free of charge would amount to forced 

In [ ]:
import os
import json

output_folder = '/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/one_shot'

os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, 'Claim_only_output.json')

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"
"Premise: {premise}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: As a matter of justice, cultural heritage and knowledge should be equally accessible to all citizens, regardless of their economic background. From behind the veil of ignorance, individuals would rationally agree to ensure that the benefits of cultural preservation are shared fairly. Making museums free of charge promotes fair equality of opportunity and allows everyone to flourish as a cultured being. By doing so, we also respect the principle of reciprocity, recognizing that cultural heritage is a social good that belongs to all.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against

Argument: While museums play a vital role in preserving and showcasing our cultural heritage, making the

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/one_shot/Claim_Premise_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise+Argumentation

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"
"Premise: {premise}"
"Argumentation: {argumentation}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: As a matter of justice, citizens have a right to access the cultural and educational resources that shape our common heritage. Behind the veil of ignorance, individuals would surely endorse institutions that make such resources available to all, regardless of economic means. By making museums free, we promote fair equality of opportunity and ensure that the benefits of cultural enrichment are not limited to the privileged few. This policy would help to foster a more informed and engaged citizenry, which is essential to a well-ordered society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against

Argument: While museums do provide valuable cultural and educational resources, making them f

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/one_shot/Claim_Premise_Argumentation_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)